# Part3 Sequence Models

Notebook version of `part3_sequence_models.py`. Run the cells from top to bottom.

In [ ]:
import math

from collections import Counter

from pathlib import Path

import pandas as pd

import torch

from torch import nn

from torch.nn.utils.rnn import pad_sequence

from torch.utils.data import DataLoader, Dataset


SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]

PAD, BOS, EOS, UNK = range(4)

STARTER_PAIRS = [
    ("i am a student", "je suis un etudiant"),
    ("i like deep learning", "j aime le deep learning"),
    ("the model is simple", "le modele est simple"),
    ("we train a network", "nous entrainons un reseau"),
    ("the loss decreases", "la perte diminue"),
    ("this image is clear", "cette image est claire"),
    ("the data is normalized", "les donnees sont normalisees"),
    ("the sequence is short", "la sequence est courte"),
    ("the result is good", "le resultat est bon"),
    ("i read the report", "je lis le rapport"),
    ("we compare models", "nous comparons les modeles"),
    ("the student tests pytorch", "l etudiant teste pytorch"),
    ("the network learns patterns", "le reseau apprend des motifs"),
    ("the classifier predicts a class", "le classifieur predit une classe"),
    ("the decoder generates words", "le decodeur genere des mots"),
    ("the encoder reads the sentence", "l encodeur lit la phrase"),
    ("gradient clipping stabilizes training", "le gradient clipping stabilise l entrainement"),
    ("beam search explores candidates", "beam search explore des candidats"),
    ("a recurrent model uses memory", "un modele recurrent utilise la memoire"),
    ("the experiment is reproducible", "l experience est reproductible"),
]


## Shared utilities

Embedded so this notebook runs without `utils.py`.


In [ ]:
"""Utilitaires communs aux trois parties du projet Deep Learning."""

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch


PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"

for directory in (FIGURES_DIR, METRICS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def get_device():
    """Return the best device available to PyTorch."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def plot_history(history, title, output_path):
    plt.figure(figsize=(7, 4))
    for name, values in history.items():
        plt.plot(range(1, len(values) + 1), values, label=name.replace("_", " "))
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def plot_confusion_matrix(matrix, class_names, title, output_path):
    plt.figure(figsize=(5, 4))
    plt.imshow(matrix, interpolation="nearest", cmap="Blues")
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=30, ha="right")
    plt.yticks(ticks, class_names)
    threshold = matrix.max() / 2 if matrix.size else 0
    for row, column in np.ndindex(matrix.shape):
        plt.text(
            column,
            row,
            str(matrix[row, column]),
            ha="center",
            va="center",
            color="white" if matrix[row, column] > threshold else "black",
        )
    plt.ylabel("Classe reelle")
    plt.xlabel("Classe predite")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def _json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Type non serialisable: {type(value).__name__}")


def save_json(payload, output_path):
    with Path(output_path).open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2, default=_json_default)



In [ ]:
def simple_tokenize(text):
    return text.lower().strip().split()


In [ ]:
def load_parallel_pairs(path="data/fra-eng/fra.txt", limit=400):
    file_path = Path(path)
    if not file_path.exists():
        return STARTER_PAIRS

    pairs = []
    with file_path.open("r", encoding="utf-8") as file:
        for line in file:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                source = " ".join(simple_tokenize(parts[0]))
                target = " ".join(simple_tokenize(parts[1]))
                if 1 <= len(source.split()) <= 8 and 1 <= len(target.split()) <= 10:
                    pairs.append((source, target))
            if len(pairs) >= limit:
                break
    return pairs if pairs else STARTER_PAIRS


In [ ]:
class Vocab:
    def __init__(self, texts, min_freq=1):
        counter = Counter()
        for text in texts:
            counter.update(simple_tokenize(text))
        self.itos = list(SPECIALS)
        for token, freq in counter.items():
            if freq >= min_freq and token not in self.itos:
                self.itos.append(token)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}

    def encode(self, text, add_bos=False, add_eos=True):
        ids = [self.stoi.get(token, UNK) for token in simple_tokenize(text)]
        if add_bos:
            ids = [BOS] + ids
        if add_eos:
            ids = ids + [EOS]
        return torch.tensor(ids, dtype=torch.long)

    def decode(self, ids):
        tokens = []
        for idx in ids:
            if idx in [PAD, BOS]:
                continue
            if idx == EOS:
                break
            tokens.append(self.itos[idx] if idx < len(self.itos) else "<unk>")
        return " ".join(tokens)

    def __len__(self):
        return len(self.itos)


In [ ]:
class LanguageDataset(Dataset):
    def __init__(self, sentences, vocab):
        self.samples = [vocab.encode(sentence, add_bos=True, add_eos=True) for sentence in sentences]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return sample[:-1], sample[1:]


In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.samples = [
            (src_vocab.encode(src, add_bos=True, add_eos=True), tgt_vocab.encode(tgt, add_bos=True, add_eos=True))
            for src, tgt in pairs
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


In [ ]:
def collate_lm(batch):
    x, y = zip(*batch)
    return (
        pad_sequence(x, batch_first=True, padding_value=PAD),
        pad_sequence(y, batch_first=True, padding_value=PAD),
    )


In [ ]:
def collate_translation(batch):
    src, tgt = zip(*batch)
    return (
        pad_sequence(src, batch_first=True, padding_value=PAD),
        pad_sequence(tgt, batch_first=True, padding_value=PAD),
    )


In [ ]:
class RecurrentLanguageModel(nn.Module):
    def __init__(self, vocab_size, cell_type="rnn", emb_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        if cell_type == "rnn":
            self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        elif cell_type == "lstm":
            self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        elif cell_type == "gru":
            self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        else:
            raise ValueError(cell_type)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(self.embedding(x))
        return self.output(out)


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        _, hidden = self.gru(self.embedding(src))
        return hidden


In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token, hidden):
        out, hidden = self.gru(self.embedding(token), hidden)
        return self.output(out), hidden


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size):
        super().__init__()
        self.encoder = Encoder(src_vocab_size)
        self.decoder = Decoder(tgt_vocab_size)

    def forward(self, src, tgt):
        hidden = self.encoder(src)
        logits = []
        for step in range(tgt.shape[1] - 1):
            token = tgt[:, step:step + 1]
            logit, hidden = self.decoder(token, hidden)
            logits.append(logit)
        return torch.cat(logits, dim=1)


In [ ]:
def train_language_model(model, loader, device, epochs=35, lr=1e-3, clip=None):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": []}
    model.to(device)
    last_grad_norm = 0.0

    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        total_tokens = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            loss.backward()
            if clip is not None:
                last_grad_norm = float(nn.utils.clip_grad_norm_(model.parameters(), clip))
            optimizer.step()
            tokens = (y != PAD).sum().item()
            total_loss += loss.item() * tokens
            total_tokens += tokens
        history["train_loss"].append(total_loss / max(total_tokens, 1))
    return history, last_grad_norm


In [ ]:
def evaluate_lm(model, loader, device):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1)).item()
            total_tokens += (y != PAD).sum().item()
    avg_loss = total_loss / max(total_tokens, 1)
    return avg_loss, math.exp(min(avg_loss, 20))


In [ ]:
def train_seq2seq(model, loader, device, epochs=120, lr=1e-3):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": []}
    model.to(device)

    for _ in range(epochs):
        model.train()
        total = 0.0
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()
            logits = model(src, tgt)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item()
        history["train_loss"].append(total / max(len(loader), 1))
    return history


In [ ]:
def greedy_decode(model, src_sentence, src_vocab, tgt_vocab, device, max_len=12):
    model.eval()
    src = src_vocab.encode(src_sentence, add_bos=True, add_eos=True).unsqueeze(0).to(device)
    with torch.no_grad():
        hidden = model.encoder(src)
        token = torch.tensor([[BOS]], device=device)
        output_ids = []
        for _ in range(max_len):
            logits, hidden = model.decoder(token, hidden)
            token = logits[:, -1:].argmax(dim=-1)
            next_id = token.item()
            output_ids.append(next_id)
            if next_id == EOS:
                break
    return tgt_vocab.decode(output_ids)


In [ ]:
def beam_search_decode(model, src_sentence, src_vocab, tgt_vocab, device, beam_width=3, max_len=12):
    model.eval()
    src = src_vocab.encode(src_sentence, add_bos=True, add_eos=True).unsqueeze(0).to(device)
    with torch.no_grad():
        hidden = model.encoder(src)
        beams = [([BOS], 0.0, hidden)]
        for _ in range(max_len):
            candidates = []
            for ids, score, h in beams:
                if ids[-1] == EOS:
                    candidates.append((ids, score, h))
                    continue
                token = torch.tensor([[ids[-1]]], device=device)
                logits, next_hidden = model.decoder(token, h)
                log_probs = torch.log_softmax(logits[:, -1, :], dim=-1)
                top_scores, top_ids = torch.topk(log_probs, beam_width)
                for s, idx in zip(top_scores[0], top_ids[0]):
                    candidates.append((ids + [idx.item()], score + s.item(), next_hidden))
            beams = sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]
        return tgt_vocab.decode(beams[0][0])


In [ ]:
def unigram_bleu(reference, candidate):
    ref_tokens = simple_tokenize(reference)
    cand_tokens = simple_tokenize(candidate)
    if not cand_tokens:
        return 0.0
    overlap = sum((Counter(cand_tokens) & Counter(ref_tokens)).values())
    precision = overlap / len(cand_tokens)
    brevity = min(1.0, math.exp(1 - len(ref_tokens) / max(len(cand_tokens), 1)))
    return brevity * precision


In [ ]:
def run_part3():
    device = get_device()
    pairs = load_parallel_pairs()
    source_texts = [src for src, _ in pairs]
    target_texts = [tgt for _, tgt in pairs]
    src_vocab = Vocab(source_texts)
    tgt_vocab = Vocab(target_texts)

    lm_dataset = LanguageDataset(target_texts, tgt_vocab)
    lm_loader = DataLoader(lm_dataset, batch_size=8, shuffle=True, collate_fn=collate_lm)

    lm_rows = []
    for cell_type in ["rnn", "lstm", "gru"]:
        model = RecurrentLanguageModel(len(tgt_vocab), cell_type=cell_type)
        history, grad_norm = train_language_model(model, lm_loader, device, clip=1.0)
        loss, ppl = evaluate_lm(model, lm_loader, device)
        lm_rows.append({
            "model": cell_type.upper(),
            "loss": loss,
            "perplexity": ppl,
            "last_grad_norm_before_clipping": grad_norm,
        })
        if cell_type == "gru":
            plot_history(history, "Partie III - apprentissage GRU LM", FIGURES_DIR / "part3_gru_lm_history.png")

    translation_dataset = TranslationDataset(pairs, src_vocab, tgt_vocab)
    translation_loader = DataLoader(translation_dataset, batch_size=8, shuffle=True, collate_fn=collate_translation)
    seq2seq = Seq2Seq(len(src_vocab), len(tgt_vocab))
    seq_history = train_seq2seq(seq2seq, translation_loader, device)
    plot_history(seq_history, "Partie III - apprentissage Seq2Seq", FIGURES_DIR / "part3_seq2seq_history.png")

    examples = []
    for src, tgt in pairs[:5]:
        greedy = greedy_decode(seq2seq, src, src_vocab, tgt_vocab, device)
        beam = beam_search_decode(seq2seq, src, src_vocab, tgt_vocab, device)
        examples.append({
            "source": src,
            "reference": tgt,
            "greedy": greedy,
            "beam_search": beam,
            "bleu_greedy": unigram_bleu(tgt, greedy),
            "bleu_beam": unigram_bleu(tgt, beam),
        })

    pd.DataFrame(lm_rows).to_csv(METRICS_DIR / "part3_language_model_results.csv", index=False)
    pd.DataFrame(examples).to_csv(METRICS_DIR / "part3_seq2seq_examples.csv", index=False)
    save_json({
        "num_pairs": len(pairs),
        "source_vocab_size": len(src_vocab),
        "target_vocab_size": len(tgt_vocab),
        "dataset_note": "Utilise data/fra-eng/fra.txt si disponible, sinon corpus integre de demonstration.",
    }, METRICS_DIR / "part3_summary.json")
    print(pd.DataFrame(lm_rows))
    print(pd.DataFrame(examples))


In [ ]:
run_part3()
